# Wahania wskazań a stan silosu

Czy im więcej materiału w silosie, tym bardziej wskazania się wahają. Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`, fragmenty bez zdarzeń bez zakłóceń; obliczenia w `dms_analysis.silo_load`, wahnięcia i odcinki z `dms_analysis.weight_changes`.

Fragmenty bez zdarzeń pocięte na godziny zegarowe (godziny krótsze niż 30 min pominięte). Dla każdej godziny:
- **stan silosu** = mediana wskazania w tej godzinie,
- **wahnięcia na godzinę** (mediana ruchoma 10 s, próg ≥ 40 kg i ≥ 160 kg; liczone wg początku wahnięcia),
- **wielkość wahnięcia** ≥ 40 kg (mediana zmiany wskazania),
- **odchylenie od prostej** w odcinkach 60 s (mediana), czyli szum na skali odsypu.

**Pułapka:** pora dnia też zmienia wahania (w dzień są większe), a stan silosu zmienia się głównie w dzień. Dlatego wszystko jest pokazane osobno dla nocy, dnia i pozostałych godzin. Druga pułapka: godziny z jednego długiego fragmentu (np. fragment 18 ma kilkadziesiąt godzin przy prawie tym samym stanie) nie są niezależne, więc korelacja jest policzona też z jednym punktem na fragment.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.between_events_fragments import prepare_samples
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.silo_load import (
    MEASURES,
    correlation_table,
    fragment_medians,
    hour_chunks,
    load_bin_table,
    load_figure,
    timeline_figure,
)
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.weight_changes import span_residuals, swings

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)

chunks = hour_chunks(samples, swings(samples), span_residuals(samples))

## Wykres: każda godzina jako punkt

Oś X = stan silosu, na kolejnych wykresach miary wahań. Kolor = pora dnia. Po najechaniu na punkt widać numer fragmentu i godzinę, więc można go obejrzeć w przeglądarce wahnięć w `weight_changes.ipynb`.

**Jak czytać:** jeśli obciążenie zwiększa wahania, punkty jednego koloru powinny iść w górę w prawo.

In [ ]:
load_figure(chunks, FIGURE_WIDTH).show()

## Korelacja

Korelacja rang (Spearmana): +1 = im więcej w silosie, tym zawsze większa miara; 0 = brak związku; −1 = odwrotnie. Nie zakłada zależności liniowej. W nawiasie liczba punktów. Przy kilkunastu punktach wartości poniżej ok. 0,5 mogą być przypadkowe.

Pierwsza tabela: punkt = godzina. Druga: punkt = fragment (mediana z jego godzin danej pory dnia).

In [ ]:
display(correlation_table(chunks).round(2))
display(correlation_table(fragment_medians(chunks)).round(2))

## Mediany w przedziałach stanu silosu

Dla każdej miary osobna mała tabela: wiersz = przedział stanu silosu co 5 t, kolumny = pora dnia, ostatnia kolumna = ile godzin wpadło do przedziału (wszystkie pory razem). Przedziały z 1–2 godzinami nic nie mówią.

In [ ]:
for measure in MEASURES:
    display(load_bin_table(chunks, measure).round(1).style.set_caption(measure).format(precision=1))

## Kontrola: stan silosu i wahnięcia w czasie

Wskazania (mediana minutowa) z zaznaczonym stanem silosu każdej godziny i liczba wahnięć ≥ 40 kg na godzinę. Pozwala sprawdzić, czy „więcej wahań przy pełnym silosie” nie jest po prostu „więcej wahań w tych konkretnych dniach”.

In [ ]:
timeline_figure(samples, chunks, FIGURE_WIDTH).show()